# Graph Building from Crystal Structures

This notebook covers:
1. Loading material IDs from CSV
2. Fetching crystal structures from Materials Project API
3. Building neighbor lists and graph representations
4. Creating graph data structures with node and edge features

**Purpose**: Convert crystal structures into graph representations suitable for GNN training.

## Part 1: Load Material IDs from CSV

In [1]:
# Import required libraries
import pandas as pd
import json
import numpy as np
from tqdm import tqdm

In [2]:
# Read the CSV file generated from Notebook 02
import os
csv_path = "cgcnn_dataset.csv"

df_train = pd.read_csv(csv_path)
print(f"CSV columns: {df_train.columns.tolist()}")
print(f"Loaded {len(df_train)} materials.")


CSV columns: ['material_id', 'cif_file', 'formula', 'formation_energy_per_atom', 'energy_above_hull', 'band_gap', 'density', 'volume', 'target']
Loaded 1000 materials.


## Part 2: Build Neighbor Lists Using ASE

We'll use ASE (Atomic Simulation Environment) to compute neighbor lists efficiently.

In [3]:
# Import required libraries
from pymatgen.core import Structure
from pymatgen.io.ase import AseAtomsAdaptor
from ase.neighborlist import neighbor_list

In [4]:
# Set cutoff radius for neighbor detection (in Angstroms)
cutoff_radius = 4.0

print(f"Using cutoff radius: {cutoff_radius} Å")

Using cutoff radius: 4.0 Å


### Example: Build Neighbor List for First Structure

In [5]:
# Grab the first structure from the downloaded CIFs
cif_file = df_train["cif_file"].iloc[1]
cif_path = os.path.join("cif_structures", cif_file)

# Load using pymatgen
pmg_struct = Structure.from_file(cif_path)

print(f"Material: {df_train['material_id'].iloc[1]}")
print(f"Formula: {df_train['formula'].iloc[1]}")
print(f"Number of atoms: {len(pmg_struct.sites)}")


Material: mp-862683
Formula: Ac2GaCu
Number of atoms: 4


In [6]:
# Convert to ASE Atoms
atoms = AseAtomsAdaptor.get_atoms(pmg_struct)
print(f"ASE Atoms object created with {len(atoms)} atoms")
print(f"Cell parameters: {atoms.get_cell_lengths_and_angles()}")

ASE Atoms object created with 4 atoms
Cell parameters: [ 5.47257855  5.47257855  5.47257855 60.         60.         60.        ]


C:\Users\Prabhat\AppData\Local\Temp\ipykernel_28168\1682769801.py:4: DeprecationWarning: Please use atoms.cell.cellpar() instead
  print(f"Cell parameters: {atoms.get_cell_lengths_and_angles()}")


In [7]:
# Build neighbor list
# Returns: i (source), j (neighbor), d (distances in Å)
edge_src, edge_dst, edge_len = neighbor_list(
    "ijd", 
    atoms, 
    cutoff=cutoff_radius, 
    self_interaction=False
)

print(f"Total neighbor pairs: {len(edge_src)}")

Total neighbor pairs: 56


### Inspect Neighbor Pairs

In [8]:
# Get chemical symbols for all atoms
symbols = atoms.get_chemical_symbols()
print(f"Elements present: {set(symbols)}")
print(f"Atomic symbols: {symbols}")

Elements present: {'Ga', 'Ac', 'Cu'}
Atomic symbols: ['Ac', 'Ac', 'Ga', 'Cu']


In [9]:
# Print first few neighbor pairs with element names
print("\nFirst 20 neighbor pairs:")
print(f"{'Edge':<6} {'Source':<15} {'Target':<15} {'Distance (Å)':>12}")
print("-" * 50)

for i in range(min(20, len(edge_src))):
    src_idx = edge_src[i]
    dst_idx = edge_dst[i]
    src_el = symbols[src_idx]
    dst_el = symbols[dst_idx]
    dist = edge_len[i]
    
    print(f"{i:<6} {src_idx:>2} ({src_el})<{8} {dst_idx:>2} ({dst_el})<{8} {dist:>12.3f}")


First 20 neighbor pairs:
Edge   Source          Target          Distance (Å)
--------------------------------------------------
0       0 (Ac)<8  1 (Ac)<8        3.870
1       0 (Ac)<8  1 (Ac)<8        3.870
2       0 (Ac)<8  2 (Ga)<8        3.351
3       0 (Ac)<8  3 (Cu)<8        3.351
4       0 (Ac)<8  1 (Ac)<8        3.870
5       0 (Ac)<8  1 (Ac)<8        3.870
6       0 (Ac)<8  3 (Cu)<8        3.351
7       0 (Ac)<8  2 (Ga)<8        3.351
8       0 (Ac)<8  2 (Ga)<8        3.351
9       0 (Ac)<8  3 (Cu)<8        3.351
10      0 (Ac)<8  1 (Ac)<8        3.870
11      0 (Ac)<8  1 (Ac)<8        3.870
12      0 (Ac)<8  3 (Cu)<8        3.351
13      0 (Ac)<8  2 (Ga)<8        3.351
14      1 (Ac)<8  0 (Ac)<8        3.870
15      1 (Ac)<8  3 (Cu)<8        3.351
16      1 (Ac)<8  2 (Ga)<8        3.351
17      1 (Ac)<8  0 (Ac)<8        3.870
18      1 (Ac)<8  2 (Ga)<8        3.351
19      1 (Ac)<8  0 (Ac)<8        3.870


### Distance Statistics

In [10]:
# Distance statistics
print("\n--- Distance Statistics (Å) ---")
print(f"Min distance:  {np.min(edge_len):.3f}")
print(f"Max distance:  {np.max(edge_len):.3f}")
print(f"Mean distance: {np.mean(edge_len):.3f}")
print(f"Std distance:  {np.std(edge_len):.3f}")


--- Distance Statistics (Å) ---
Min distance:  3.351
Max distance:  3.870
Mean distance: 3.573
Std distance:  0.257


In [11]:
# Neighbor count per atom (degree distribution)
unique, counts = np.unique(edge_src, return_counts=True)

print("\n--- Neighbor Count Per Atom ---")
for i, c in zip(unique, counts):
    print(f"Atom {i:>2} ({symbols[i]}): {c:>2} neighbors")

print(f"\nAverage coordination number: {np.mean(counts):.2f}")


--- Neighbor Count Per Atom ---
Atom  0 (Ac): 14 neighbors
Atom  1 (Ac): 14 neighbors
Atom  2 (Ga): 14 neighbors
Atom  3 (Cu): 14 neighbors

Average coordination number: 14.00


## Part 3: Create Graph Data Structure

Now we'll build the complete graph representation with node and edge features.

In [12]:
# Import PyTorch for tensors
import torch

In [13]:
# Load atom embeddings
with open("atom_embedding.json", "r") as f:
    ATOM_EMB = json.load(f)

EMB_LEN = ATOM_EMB["embedding_length"]
EMB_TAB = ATOM_EMB["embeddings"]

print(f"Loaded embeddings for {len(EMB_TAB)} elements")
print(f"Embedding length: {EMB_LEN}")

Loaded embeddings for 103 elements
Embedding length: 186


In [14]:
def get_atom_embedding(symbol: str):
    """Return embedding vector for given atomic symbol."""
    vec = EMB_TAB.get(symbol)
    if vec is None:
        return [0.0] * EMB_LEN
    return vec

### Edge Feature Encoding: Radial Basis Functions

In [15]:
def rbf(distances: np.ndarray, r_min=0.5, r_max=6.0, bins=32, gamma=None):
    """
    Compute Gaussian radial basis expansion for distances.
    Returns array (E, bins).
    """
    centers = np.linspace(r_min, r_max, bins, dtype=np.float32)
    d = distances.reshape(-1, 1)
    if gamma is None:
        spacing = (r_max - r_min) / max(1, bins - 1)
        sigma = spacing if spacing > 0 else 1.0
        gamma = 1.0 / (2.0 * sigma * sigma)
    return np.exp(-gamma * (d - centers) ** 2)

### Complete Graph Builder Function

In [16]:
def build_graph_from_row(row, cutoff=4.0, rbf_bins=32):
    """
    Convert one DataFrame row into a graph representation.
    
    Returns:
        Dictionary with keys:
        - material_id: material identifier
        - x: node features (N, F)
        - edge_index: edge indices (2, E)
        - edge_attr: edge features (E, D)
        - y: target value (scalar)
        - num_nodes: number of nodes
    """
    cif_file = row["cif_file"]
    cif_path = os.path.join("cif_structures", cif_file)
    pmg = Structure.from_file(cif_path)
    atoms = AseAtomsAdaptor.get_atoms(pmg)

    # Build neighbor list (edges)
    i_src, j_dst, d_len = neighbor_list("ijd", atoms, cutoff=cutoff, self_interaction=False)
    i_src = np.asarray(i_src, dtype=np.int64)
    j_dst = np.asarray(j_dst, dtype=np.int64)
    d_len = np.asarray(d_len, dtype=np.float32)

    # Node features from atom embeddings
    symbols = atoms.get_chemical_symbols()
    x = np.vstack([np.array(get_atom_embedding(sym), dtype=np.float32) for sym in symbols])

    # Edge features: [distance, RBF]
    edge_dist = d_len.reshape(-1, 1)
    edge_rbf = rbf(d_len, r_min=0.5, r_max=6.0, bins=rbf_bins)
    edge_attr = np.hstack([edge_dist, edge_rbf]).astype(np.float32)

    # Target value (try multiple properties)
    y = None
    for key in ["formation_energy_per_atom", "energy_above_hull", "band_gap"]:
        if key in row and row[key] is not None:
            try:
                y = float(row[key])
                break
            except:
                pass

    return {
        "material_id": row.get("material_id", None),
        "x": torch.from_numpy(x),
        "edge_index": torch.from_numpy(np.vstack([i_src, j_dst])),
        "edge_attr": torch.from_numpy(edge_attr),
        "y": None if y is None else torch.tensor([y], dtype=torch.float32),
        "num_nodes": x.shape[0],
    }

### Build Graph for Example Structure

In [17]:
# Build graph for the first material
graph_example = build_graph_from_row(df_train.iloc[0], cutoff=cutoff_radius, rbf_bins=32)

print(f"Material ID: {graph_example['material_id']}")
print(f"Number of nodes: {graph_example['num_nodes']}")
print(f"Number of edges: {graph_example['edge_index'].shape[1]}")
print(f"Node feature shape: {graph_example['x'].shape}")
print(f"Edge feature shape: {graph_example['edge_attr'].shape}")
print(f"Target value: {graph_example['y']}")

Material ID: mp-1183120
Number of nodes: 4
Number of edges: 56
Node feature shape: torch.Size([4, 186])
Edge feature shape: torch.Size([56, 33])
Target value: tensor([-0.0168])


### Build Graphs for All Materials

In [18]:
# Build graphs for all materials in the dataset
print("Building graphs for all materials...")
graphs = []

for idx, row in tqdm(df_train.iterrows(), total=len(df_train), desc="Building graphs"):
    try:
        graph = build_graph_from_row(row, cutoff=cutoff_radius, rbf_bins=32)
        graphs.append(graph)
    except Exception as e:
        print(f"\n⚠️ Error processing {row.get('material_id', idx)}: {e}")

print(f"\n✅ Successfully built {len(graphs)} graphs")

Building graphs for all materials...


Building graphs:   1%|▊                                                                                              | 8/1000 [00:00<00:12, 77.42it/s]C:\Users\Prabhat\Documents\Semester 7\CO4015 - Mini Project\cgcnn\.venv\Lib\site-packages\pymatgen\core\structure.py:3117: UserWarning: Issues encountered while parsing CIF: 1 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  struct = parser.parse_structures(primitive=primitive)[0]
Building graphs:   2%|█▌                                                                                            | 16/1000 [00:00<00:14, 69.58it/s]C:\Users\Prabhat\Documents\Semester 7\CO4015 - Mini Project\cgcnn\.venv\Lib\site-packages\pymatgen\core\structure.py:3117: UserWarning: Issues encountered while parsing CIF: 4 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  struct = parser.parse_structures(primitive=primitive)[0]
Building graphs:   2%|██▎                                 


✅ Successfully built 1000 graphs


### Save Graph Dataset

In [19]:
# Save graphs using PyTorch
torch.save(graphs, "graph_dataset.pt")
print("✅ Saved graph dataset to 'graph_dataset.pt'")

✅ Saved graph dataset to 'graph_dataset.pt'


### Dataset Statistics

In [20]:
# Compute dataset statistics
num_nodes_list = [g['num_nodes'] for g in graphs]
num_edges_list = [g['edge_index'].shape[1] for g in graphs]

print("\n=== Dataset Statistics ===")
print(f"Total graphs: {len(graphs)}")
print(f"\nNodes per graph:")
print(f"  Min:  {np.min(num_nodes_list)}")
print(f"  Max:  {np.max(num_nodes_list)}")
print(f"  Mean: {np.mean(num_nodes_list):.2f}")
print(f"\nEdges per graph:")
print(f"  Min:  {np.min(num_edges_list)}")
print(f"  Max:  {np.max(num_edges_list)}")
print(f"  Mean: {np.mean(num_edges_list):.2f}")


=== Dataset Statistics ===
Total graphs: 1000

Nodes per graph:
  Min:  1
  Max:  180
  Mean: 28.00

Edges per graph:
  Min:  0
  Max:  3620
  Mean: 483.12


## Summary

We have successfully:
1. ✅ Loaded material IDs from CSV
2. ✅ Downloaded crystal structures from Materials Project
3. ✅ Built neighbor lists using ASE
4. ✅ Created graph representations with node and edge features
5. ✅ Saved graph dataset for GNN training

The graphs are now ready to be used for training graph neural networks!